In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATH
# ============================================================

results_dir = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset fiben execution/results"
)

agg = pd.read_csv(results_dir / "benchmark_aggregate_results.csv")

# ============================================================
# EXTRACT QUERY ID (Q1, Q2, ..., Q10)
# ============================================================

agg["official_id"] = agg["query_name"].str.extract(r"^(Q\d+)")

# ============================================================
# QUERY GROUP CLASSIFICATION (FIBEN)
# ============================================================

def query_group(qid):
    if qid in ["Q1", "Q2"]:
        return "lookup"
    if qid in ["Q3", "Q4", "Q5", "Q6"]:
        return "complex_read"
    if qid in ["Q7", "Q8", "Q9"]:
        return "aggregation"
    if qid == "Q10":
        return "update"
    return "other"

agg["query_group"] = agg["official_id"].apply(query_group)

# ============================================================
# FILTER HOT RUNS
# ============================================================

hot = agg[agg["run_phase"] == "hot"].copy()

# ============================================================
# MAIN ANALYSIS
# ============================================================

rows = []

# ⚠️ TOTAL POSSIBLE CONFIGS (G0–G9)
n_C = 10

for query_name, grp in hot.groupby("query_name"):

    # ---------------------------
    # BEST OVERALL (ground truth)
    # ---------------------------
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]

    # ---------------------------
    # ACTIVATED SET (≠ control)
    # ---------------------------
    activated = grp[grp["final_benchmark_group"] != "control"]

    # ---------------------------
    # PRIMARY ONLY
    # ---------------------------
    primary = grp[grp["final_benchmark_group"] == "primary"]

    # ---------------------------
    # BEST IN ACTIVATED
    # ---------------------------
    best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]

    # ---------------------------
    # BEST PRIMARY
    # ---------------------------
    if len(primary) > 0:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        primary_regret = (
            best_primary["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"]
    else:
        best_primary = None
        primary_regret = np.nan

    # ---------------------------
    # DSR (Design Space Reduction)
    # ---------------------------
    n_A = activated["candidate_id"].nunique()
    dsr = 1 - (n_A / n_C)

    # ---------------------------
    # BUILD ROW
    # ---------------------------
    rows.append({
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "query_group": best_all["query_group"],

        "n_tested_configs": grp["candidate_id"].nunique(),
        "n_activated_configs": n_A,
        "DSR": dsr,

        "best_config": best_all["g_class"],
        "best_group": best_all["final_benchmark_group"],
        "best_design_pattern": best_all["design_pattern"],
        "best_p95_ms": best_all["p95_latency_ms"],

        "top1_preserved_by_activated": (
            best_activated["candidate_id"] == best_all["candidate_id"]
        ),

        "activated_regret": (
            best_activated["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"],

        "best_primary_config": None if best_primary is None else best_primary["g_class"],
        "best_primary_p95_ms": None if best_primary is None else best_primary["p95_latency_ms"],
        "primary_regret": primary_regret,
    })

# ============================================================
# FINAL DATAFRAME
# ============================================================

analysis_df = pd.DataFrame(rows).sort_values("official_id")

display(analysis_df)

# ============================================================
# GLOBAL METRICS (SchemaLens style)
# ============================================================

print("\n=== GLOBAL METRICS ===")
print("Average DSR:", analysis_df["DSR"].mean())
print("Top-1 preservation activated:", analysis_df["top1_preserved_by_activated"].mean())
print("Mean activated regret:", analysis_df["activated_regret"].mean())
print("Mean primary regret:", analysis_df["primary_regret"].dropna().mean())

# ============================================================
# SECONDARY-AFFECTED CASES (IMPORTANT INSIGHT)
# ============================================================

print("\n=== QUERIES WHERE BEST = SECONDARY_AFFECTED ===")

display(
    analysis_df[
        analysis_df["best_group"] == "secondary_affected"
    ][
        [
            "official_id",
            "query_name",
            "query_group",
            "best_config",
            "best_design_pattern",
            "best_p95_ms",
            "best_primary_config",
            "best_primary_p95_ms",
            "primary_regret",
        ]
    ]
)

,official_id,query_name,query_group,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret
1,Q1,Q1_CompanyProfileIBM,lookup,2,1,0.9,CONTROL,control,normalized_reference_baseline,0.233373,False,1.390352,G0,0.557844,1.390352
0,Q10,Q10_CreateAccountHoldingAndBuyTransaction,update,7,6,0.4,G9,secondary_affected,benchmark_tradeoff_alternative,0.575564,True,0.000000,G4,0.685536,0.191069
2,Q2,Q2_CompanyWithIndustryCountryAndListedSecurities,lookup,6,5,0.5,G5,primary,shared_target_reference_strategy,0.117117,True,0.000000,G5,0.117117,0.000000
3,Q3,Q3_SecuritiesHeldInEachFinancialServiceAccount,complex_read,7,6,0.4,G5,primary,shared_target_reference_strategy,0.434946,True,0.000000,G5,0.434946,0.000000
4,Q4,Q4_CompaniesReachedFromPersonThroughAccountHol...,complex_read,7,6,0.4,CONTROL,control,normalized_reference_baseline,1.002833,False,0.180708,G4,1.184053,0.180708
5,Q5,Q5_ReportsAndMetricDataOfCompany,complex_read,4,3,0.7,G2,primary,embedded_containment,37.626791,True,0.000000,G2,37.626791,0.000000
6,Q6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,complex_read,6,5,0.5,G9,secondary_affected,benchmark_tradeoff_alternative,0.385924,True,0.000000,G1,0.419557,0.087151
7,Q7,Q7_PersonsWhoBoughtMoreIBMThanSold,aggregation,8,7,0.3,G4,primary,deep_nested_document,1.186999,True,0.000000,G4,1.186999,0.000000
8,Q8,Q8_IBMTransactionsBelowAverageSellingPrice,aggregation,6,5,0.5,G7,secondary_affected,update_aware_reference_design,0.863551,True,0.000000,G3,1.413242,0.636547
9,Q9,Q9_PersonsWhoBoughtAndSoldSameStock,aggregation,7,6,0.4,G7,secondary_affected,update_aware_reference_design,0.851476,True,0.000000,G5,0.917673,0.077744



=== GLOBAL METRICS ===
Average DSR: 0.5000000000000001
Top-1 preservation activated: 0.8
Mean activated regret: 0.15710601267973692
Mean primary regret: 0.25635704143291915

=== QUERIES WHERE BEST = SECONDARY_AFFECTED ===


,official_id,query_name,query_group,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
0,Q10,Q10_CreateAccountHoldingAndBuyTransaction,update,G9,benchmark_tradeoff_alternative,0.575564,G4,0.685536,0.191069
6,Q6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,complex_read,G9,benchmark_tradeoff_alternative,0.385924,G1,0.419557,0.087151
8,Q8,Q8_IBMTransactionsBelowAverageSellingPrice,aggregation,G7,update_aware_reference_design,0.863551,G3,1.413242,0.636547
9,Q9,Q9_PersonsWhoBoughtAndSoldSameStock,aggregation,G7,update_aware_reference_design,0.851476,G5,0.917673,0.077744
